In [1]:
from __future__ import print_function, division
import os
import numpy as np
import raytrace as rt
from raytrace import implane
import rtcore

# --------------------------------------------------
# OUTPUT DIRECTORY
# --------------------------------------------------
SIM_DIR = "/home/hp/raytrace_work/raytrace_results/Simulation_Data_201x201"
if not os.path.exists(SIM_DIR):
    os.makedirs(SIM_DIR)
print("Simulation data: {}".format(SIM_DIR))

# --------------------------------------------------
# 15 FREQUENCIES
# --------------------------------------------------
freq_list_raw = np.linspace(15.e6, 300.e6, 20)
freq_list     = np.array([round(f / 1.e6) * 1.e6
                           for f in freq_list_raw])

print("\nFrequencies:")
for i, f in enumerate(freq_list):
    print("  [{:02d}]  {:.0f} MHz".format(i+1, f/1.e6))

# --------------------------------------------------
# PARAMETERS
# --------------------------------------------------
grid    = (201, 201)
rect    = (-2, -2, 2, 2)
obs     = (215, 0, 0)
rsph    = 25
niter   = 1500
nx, ny  = int(grid[0]), int(grid[1])

Simulation data: /home/hp/raytrace_work/raytrace_results/Simulation_Data_201x201

Frequencies:
  [01]  15 MHz
  [02]  30 MHz
  [03]  45 MHz
  [04]  60 MHz
  [05]  75 MHz
  [06]  90 MHz
  [07]  105 MHz
  [08]  120 MHz
  [09]  135 MHz
  [10]  150 MHz
  [11]  165 MHz
  [12]  180 MHz
  [13]  195 MHz
  [14]  210 MHz
  [15]  225 MHz
  [16]  240 MHz
  [17]  255 MHz
  [18]  270 MHz
  [19]  285 MHz
  [20]  300 MHz


/home/hp/miniconda3/envs/py27/lib/python2.7/site-packages/pyfits/__init__.py:22: PyFITSDeprecationWarning: PyFITS is deprecated, please use astropy.io.fits
  PyFITSDeprecationWarning)  # noqa


In [2]:
# --------------------------------------------------
# ALL 10000 RAYS
# --------------------------------------------------
trkrays_ALL = []
for i in range(nx):
    for j in range(ny):
        trkrays_ALL.append([i, j])
print("\nTotal rays: {}".format(len(trkrays_ALL)))

# --------------------------------------------------
# SUBSET A: 25 rays (5x5 grid)
# indices: 0,25,50,75,99 in both i and j
# --------------------------------------------------
pos_A = [0, 25, 50, 75, 99]
subset_A = []
for i in pos_A:
    for j in pos_A:
        subset_A.append(i * ny + j)   # global index

# --------------------------------------------------
# SUBSET B: 10 rays along i=50 (center row)
# j: 0,10,20,30,40,50,60,70,80,90
# --------------------------------------------------
pos_B_j  = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
subset_B = [50 * ny + j for j in pos_B_j]

print("Subset A: {} rays".format(len(subset_A)))
print("Subset B: {} rays".format(len(subset_B)))

# --------------------------------------------------
# SAVE PARAMS
# --------------------------------------------------
params = {
    'grid'     : grid,
    'rect'     : rect,
    'obs'      : obs,
    'rsph'     : rsph,
    'niter'    : niter,
    'freq_list': freq_list.tolist(),
    'subset_A' : subset_A,
    'subset_B' : subset_B,
    'nx'       : nx,
    'ny'       : ny
}
np.save(os.path.join(SIM_DIR, "sim_params.npy"), params)
print("Saved params.")

# ==================================================
# SIMULATE
# ==================================================
for freq_idx, freq_hz in enumerate(freq_list):

    mhz = int(round(freq_hz / 1.e6))
    print("\n[{:02d}/15] {} MHz".format(freq_idx+1, mhz))

    sim = rt.implane(
        grid, rect, obs, rsph,
        freq=freq_hz,
        mode='TbrIQUV',
        trkparms=['pos'],
        trknpmax=10000
    )
    sim.package = '/home/hp/raytrace/py_raytr_threaded'
    sim.trace(niter, trkrays_ALL)

    traj    = sim.traj.pos
    tbriquv = np.array(sim.tbriquv)

    print("  traj shape   : {}".format(traj.shape))
    print("  tbriquv shape: {}".format(tbriquv.shape))

    np.save(os.path.join(
        SIM_DIR, "traj_{}mhz.npy".format(mhz)), traj)
    np.save(os.path.join(
        SIM_DIR, "tbriquv_{}mhz.npy".format(mhz)), tbriquv)

    del traj, tbriquv

print("\nAll simulations done.")


Total rays: 40401
Subset A: 25 rays
Subset B: 10 rays
Saved params.

[01/15] 15 MHz
plf_cname =  plasma_parameters.c
fname =  plasma_parameters.c
dname =  /home/hp/raytrace_work/raytrace_scripts
bfname =  plasma_parameters.c
name =  plasma_parameters
gcc -g -fPIC  -I/home/hp/miniconda3/envs/py27/lib/python2.7/site-packages/raytrace/inc -c /home/hp/raytrace_work/raytrace_scripts/streamer.c -o streamer.o
gcc -g -fPIC  -I/home/hp/miniconda3/envs/py27/lib/python2.7/site-packages/raytrace/inc -c plasma_parameters.c -o plasma_parameters.o
gcc -shared streamer.o plasma_parameters.o -L/home/hp/miniconda3/envs/py27/lib/python2.7/site-packages/raytrace/lib -L/usr/lib -lm -lmxv -o plasma_parameters.so
  traj shape   : (40401, 1000, 3)
  tbriquv shape: (201, 201, 4)

[02/15] 30 MHz
plf_cname =  plasma_parameters.c
fname =  plasma_parameters.c
dname =  /home/hp/raytrace_work/raytrace_scripts
bfname =  plasma_parameters.c
name =  plasma_parameters
gcc -g -fPIC  -I/home/hp/miniconda3/envs/py27/lib/p